# 01 · Features and dataset

Checks that the feature engineering (`features.py`) and the dataset builder (`dataset.py`) are correct,
then builds (or loads) the processed dataset used for training.

Run top to bottom after every kernel restart (**Restart & Run All**).

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, glob
import numpy as np
import torch

TASK_DIR = "/home/yousef/ucl_ws/comp0245-final/robotnik_arm_task"
RAW_DIR  = os.path.join(TASK_DIR, "data", "raw")
PROC     = os.path.join(TASK_DIR, "data", "processed_step10.npz")
CKPT     = os.path.join(TASK_DIR, "checkpoints")
sys.path.insert(0, os.path.join(TASK_DIR, "yousef"))

from features import *
from dataset import split_episodes, build_dataset

## 1. Feature checks on one episode

| Check | Expected | Result obtained (2 Nov) |
|---|---|---|
| `world_to_base` — robot facing +y, vector +y | `[[1, 0]]` | `[[1.0, 6.1e-17]]` ✅ |
| max goal distance in phase 3 (`ep_00000`) | < ~0.08 m | 0.0706 m ✅ |
| wheel round trip `vw_to_wheels(wheels_to_vw(w)) - w` | ~1e-6 | 2.4e-7 ✅ |
| shapes / NaNs | same N, no NaN | (14061, 10) (14061, 25) ✅ |

In [2]:
print(world_to_base(np.array([[0., 1.]]), np.array([np.pi / 2])))   # expect [[1, 0]]

[[1.000000e+00 6.123234e-17]]


In [3]:
d = np.load(os.path.join(RAW_DIR, "ep_00000.npz"))
Xb, nb = base_features(d)
Xa, na = arm_features(d)
print("max goal dist in phase 3:", Xb[d["phase"] == 3, 2].max())     # expect < ~0.08
w = d["wheel_cmds"]
print("wheel round trip error:", np.abs(vw_to_wheels(wheels_to_vw(w)) - w).max())   # expect ~1e-6
print(Xb.shape, Xa.shape, len(d["t"]), np.isnan(Xb).any(), np.isnan(Xa).any())

max goal dist in phase 3: 0.07061196
wheel round trip error: 2.3841858e-07
(14061, 10) (14061, 25) 14061 False False


## 2. Goal distance when the arm starts reaching (first 100 episodes)

Finding (2 Nov): max 0.161 m, 32/100 episodes above 0.08 m, **none timed out** (`nav_time` < 25 s).
Causes in the expert: the overshoot rule in `base_controller` (`passed_goal`: finish driving within 0.15 m
if the goal is behind) + drift while turning on the spot / arm reaction forces.
→ "base done" is not a single distance threshold, which justifies giving ArmNet distance + bearing + speed.

In [4]:
files = sorted(glob.glob(os.path.join(RAW_DIR, "ep_0*.npz")))[:100]
print(len(files), "files")                                            # expect 100
for f in files:
    e = np.load(f)
    if e["success"]:
        m = base_features(e)[0][e["phase"] == 3, 2].max()
        if m > 0.08:
            print(os.path.basename(f), f"max dist={m:.3f}", f"nav_time={float(e['nav_time']):.1f}s")

100 files
ep_00009.npz max dist=0.080 nav_time=8.6s
ep_00013.npz max dist=0.091 nav_time=5.1s
ep_00015.npz max dist=0.107 nav_time=4.6s
ep_00016.npz max dist=0.081 nav_time=8.7s
ep_00017.npz max dist=0.091 nav_time=8.3s
ep_00018.npz max dist=0.080 nav_time=7.3s
ep_00019.npz max dist=0.081 nav_time=7.0s
ep_00020.npz max dist=0.085 nav_time=6.6s
ep_00022.npz max dist=0.089 nav_time=6.6s
ep_00023.npz max dist=0.087 nav_time=4.0s
ep_00025.npz max dist=0.110 nav_time=4.0s
ep_00029.npz max dist=0.080 nav_time=5.0s
ep_00030.npz max dist=0.082 nav_time=6.5s
ep_00031.npz max dist=0.110 nav_time=6.9s
ep_00033.npz max dist=0.158 nav_time=6.3s
ep_00034.npz max dist=0.080 nav_time=2.7s
ep_00039.npz max dist=0.080 nav_time=11.3s
ep_00040.npz max dist=0.081 nav_time=1.8s
ep_00050.npz max dist=0.103 nav_time=7.5s
ep_00054.npz max dist=0.161 nav_time=6.6s
ep_00056.npz max dist=0.138 nav_time=6.2s
ep_00057.npz max dist=0.116 nav_time=8.2s
ep_00059.npz max dist=0.093 nav_time=5.6s
ep_00061.npz max dist=0

## 3. Episode split

Successful episodes only; IDs ≥ 9000 = test; the rest split 85 / 15 **by episode** (seed 0).
Result (2 Nov): train 1008 / val 177 / test 98 = 1283, no overlap.

In [5]:
train_f, val_f, test_f = split_episodes()
print(len(train_f), len(val_f), len(test_f))                          # 1008 177 98
print(set(train_f) & set(val_f), set(train_f) & set(test_f))          # set() set()

1008 177 98
set() set()


## 4. Build or load the processed dataset

Every 10th step (1 kHz → 100 Hz), robot-frame features, normalisation stats from **train only**.
Built once (≈ 1–3 min) and saved to `data/processed_step10.npz`; afterwards it is just loaded.
Result (2 Nov): train 1,026,259 rows / val 175,714 / test 99,966, no NaNs.

In [ ]:
REBUILD = False                       # set True after changing features.py or dataset.py
if REBUILD or not os.path.exists(PROC):
    D = build_dataset(step=10)
else:
    D = dict(np.load(PROC))
for k, v in D.items():
    print(f"{k:14s} {v.shape}")
print("any NaN:", any(np.isnan(v).any() for v in D.values()))

In [6]:
PROC = os.path.join(TASK_DIR, "data", "processed_step10_noise.npz")
REBUILD = True
if REBUILD or not os.path.exists(PROC):
    D = build_dataset(step=10, use_noise=True)
else:
    D = dict(np.load(PROC))

episodes: train 1334, val 235, test 98
saved /home/yousef/ucl_ws/comp0245-final/robotnik_arm_task/data/processed_step10_noise.npz
